# 04 - Boarding Time Causal Analysis

This notebook demonstrates the Phase 2 causal evaluation framework:

1. **Basic comparison** (Mann-Whitney U)
2. **Propensity Score Matching** (selection bias control)
3. **Interrupted Time Series** (temporal causality)
4. **Heterogeneous Treatment Effects** (subgroup analysis)
5. **Spillover Analysis** (contamination check)
6. **Operational Evaluation** (Census-Staffing alignment, Tradeoff curves)

**Note:** Raw data not included. Supply your own patient-level CSV.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
from analysis.statistical_analysis import BoardingTimeAnalysis
from analysis.operational_evaluation import OperationalEvaluation

## 1. Load Patient Data

Expected columns: `arrival`, `departure`, `boarding_time`, `exam_los`,
`hospital_los`, `triage_risk`, `arrival_hour`, `day_of_week`, `month_num`, `half`

In [ ]:
PATIENT_DATA_PATH = "../data/patient_data.csv"  # <-- Replace
SHIFTS_DATA_PATH = "../data/shifts.csv"          # <-- Replace

patients = pd.read_csv(PATIENT_DATA_PATH, parse_dates=['arrival', 'departure'])
shifts = pd.read_csv(SHIFTS_DATA_PATH, parse_dates=['date'])

print(f"Patients: {len(patients)}")
print(f"Shift records: {len(shifts)}")

## 2. Basic Comparison

In [ ]:
analysis = BoardingTimeAnalysis(patients)
basic = analysis.basic_comparison()

print(f"Intervention: {basic['intervention_mean']} min")
print(f"Control:      {basic['control_mean']} min")
print(f"Difference:   {basic['difference']} min")
print(f"P-value:      {basic['p_value']:.4f}")

## 3. Propensity Score Matching

In [ ]:
psm = analysis.propensity_score_matching()

print(f"Matched pairs: {psm['n_pairs']}")
print(f"ATT: {psm['att']} min")
print(f"P-value: {psm['p_value']:.6f}")
print(f"\nBalance check (SMD):")
for cov, vals in psm['balance_smd'].items():
    status = 'OK' if vals['balanced'] else 'IMBALANCED'
    print(f"  {cov}: SMD={vals['smd_after']:.4f} [{status}]")

## 4. Interrupted Time Series

In [ ]:
its = analysis.interrupted_time_series()

print("ITS Segmented Regression Results:")
print(f"  Baseline trend:    coef={its['baseline_trend']['coef']}, p={its['baseline_trend']['p_value']}")
print(f"  Immediate effect:  coef={its['immediate_effect']['coef']}, p={its['immediate_effect']['p_value']}")
print(f"  Trend change:      coef={its['trend_change']['coef']}, p={its['trend_change']['p_value']}")
print(f"  R-squared:         {its['r_squared']}")

## 5. Heterogeneous Treatment Effects

In [ ]:
hte = analysis.heterogeneous_effects()

for group, vals in hte.items():
    sig = '*' if vals['p_value'] < 0.05 else 'NS'
    print(f"{group}: diff={vals['difference']} min, p={vals['p_value']:.4f} [{sig}]")

## 6. Spillover Analysis

In [ ]:
spillover = analysis.spillover_analysis()

print(f"Spearman r: {spillover['spearman_r']}")
print(f"P-value: {spillover['p_value']}")
print(f"No spillover: {spillover['no_spillover']}")
print(f"Interpretation: {spillover['interpretation']}")

## 7. Operational Evaluation

In [ ]:
ops_eval = OperationalEvaluation(
    patient_data=patients,
    shifts_data=shifts,
    output_dir='../docs/figures',
)

results = ops_eval.run_all(save_figures=True)

print(f"MSD Intervention: {results['msd']['msd_intervention']}")
print(f"MSD Control: {results['msd']['msd_control']}")
print(f"MSD Reduction: {results['msd']['reduction_pct']}%")
print(f"\nRegression R-squared: {results['r_squared']}")
print(f"\nRegression table:")
print(results['regression_table'].to_string(index=False))